# 10 — Opti, SQP, and quadratic-program interfaces

**Objectives.** Express the same NLP readably with `Opti`, change parameters without rebuilding, and understand where QP plugins fit. `Opti` is a modeling layer over CasADi functions and NLP solvers; it does not change the mathematics.


## Learning path and assumptions

**Prerequisites:** Chapter 09.

**Problem:** Compare Opti, lower-level NLPs, and a real quadratic-program solve.

**Assumptions and conventions:** QP objectives are quadratic and constraints affine; a nonlinear ball is not a QP constraint.

**How to read the results:** Convenient modeling changes the interface, not the underlying optimality conditions.

### Resolve the local checkout

This setup locates repository-local presentation and numerical helpers and enables autoreload. It changes Python import resolution, not the mathematical model. Run it once after restarting the kernel.

In [8]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

### Express an optimization problem with the Opti modeling layer

`ca.Opti` provides a higher-level interface for declaring variables,
parameters, objectives, and constraints. `set_value` supplies parameter
data without rebuilding the graph, while `set_initial` supplies only a
solver starting point. `ca.sumsqr` constructs a scalar sum of squared
elements and is used here for the circular constraint.

The model is

$$
\begin{aligned}
\min_x\quad &(x_0-t)^2+(x_1-1)^2\\
\text{s.t.}\quad &x_0\ge0,\;x_1\ge0,\\
&x_0+x_1\ge1,\\
&x_0^2+x_1^2\le4,
\end{aligned}
$$

where $t$ is an Opti parameter set to $2$ before solving. Although the
objective is quadratic, the quadratic inequality makes this a convenient
demonstration of the general nonlinear Opti/IPOPT workflow.


In [15]:
import casadi as ca

optimization = ca.Opti()
x = optimization.variable(2)
target = optimization.parameter()

optimization.minimize((x[0] - target) ** 2 + (x[1] - 1) ** 2)
optimization.subject_to(x >= 0)
optimization.subject_to(x[0] + x[1] >= 1)
optimization.subject_to(ca.sumsqr(x) <= 4)

optimization.set_value(target, 2.0)
optimization.set_initial(x, [0.5, 0.5])
optimization.solver(
    "fatrop",
    {"print_time": False},
    {"print_level": 0},
)

solution = optimization.solve()
print(
    "optimal x:",
    solution.value(x),
    "status:",
    solution.stats()["return_status"],
)

optimal x: [1.78885439 0.89442719] status: 0


### Inspect available conic solver interfaces

CasADi solver plugins are compiled independently. `CasadiMeta.plugins()`
reports the available backends using `Family::name` strings, so filtering
the `Conic` family reveals which quadratic/conic solvers can be selected in
the current environment. `ca.conic_in()` lists the standardized fields
accepted by CasADi's low-level conic interface.

A common quadratic-program form is

$$
\begin{aligned}
\min_x\quad &\tfrac12x^\mathsf{T}Hx+g^\mathsf{T}x\\
\text{s.t.}\quad &\ell_a\le Ax\le u_a,\\
&\ell_x\le x\le u_x.
\end{aligned}
$$

This cell does not solve that problem; it inspects which plugin backends
and standardized inputs would be available to do so.


In [16]:
# CasadiMeta.plugins reports compiled solver backends as Family::name.
installed_plugins = ca.CasadiMeta.plugins().split(";")
conic_plugins = [plugin for plugin in installed_plugins if plugin.startswith("Conic::")]
print("installed conic/QP plugins:", conic_plugins)

# conic_in lists the standardized input fields accepted by conic solvers.
print("conic input fields:", ca.conic_in())

installed conic/QP plugins: ['Conic::qpoases', 'Conic::fatrop', 'Conic::proxqp', 'Conic::osqp', 'Conic::nlpsol', 'Conic::qrqp', 'Conic::ipqp']
conic input fields: ['h', 'g', 'a', 'lba', 'uba', 'lbx', 'ubx', 'x0', 'lam_x0', 'lam_a0', 'q', 'p']


A QP has a quadratic objective and linear constraints. CasADi exposes QP solvers through `conic`/`qpsol`; availability varies by binary and license. `sqpmethod` repeatedly builds QP approximations to an NLP and is useful for comparison, while IPOPT is this course's primary backend.

`opti.debug.value(expr, opti.initial())`, `opti.debug.show_infeasibilities()`, `x_describe`, and `g_describe` connect algebra back to source. **Recap:** use `Opti` for readable modeling and raw `nlpsol` when exact vector ordering or generated interfaces matter. [Exercises](../exercises/04_nlp_debugging.ipynb).


## Solve an actual QP

The quadratic program $\min_x\frac12x^THx+g^Tx$ uses $H=2I$, $g=[-2,-4]^T$, and the affine equality $[1,1]x=2$ with $x\ge0$. Its exact optimum is $[0.5,1.5]^T$. `qpsol` inspects the symbolic quadratic and affine structure; QRQP solves the resulting QP.

In [17]:
import numpy as np
qp_x=ca.SX.sym("qp_x",2)
qp=ca.qpsol("teaching_qp","qrqp",{"x":qp_x,"f":ca.dot(qp_x,qp_x)-2*qp_x[0]-4*qp_x[1],
                                         "g":qp_x[0]+qp_x[1]},
            {"print_header":False,"print_iter":False})
qp_result=qp(lbx=0,ubx=ca.inf,lbg=2,ubg=2)
np.testing.assert_allclose(qp_result["x"].full().ravel(),[.5,1.5],atol=1e-8)
print("QP optimum:",qp_result["x"],"status:",qp.stats())

QP optimum: [0.5, 1.5] status: {'iter_count': -1, 'return_status': 'success', 'success': True, 'unified_return_status': 'SOLVER_RET_SUCCESS'}


**Checkpoint:** explain why replacing the affine equality with a circular constraint changes the problem class.

## Parameter reuse with Opti

Change solve-time data with `set_value`, keep graph structure fixed, and provide the previous primal result as an initial guess. Warm starts can reduce work, but their benefit depends on the new problem; success still requires explicit validation.

In [18]:
def opti_parameter_experiment(target_value=1.5):
    optimization.set_value(target,target_value)
    optimization.set_initial(x,solution.value(x))
    new_solution=optimization.solve()
    print("target:",target_value,"x:",new_solution.value(x),"status:",new_solution.stats()["return_status"])
    assert new_solution.stats()["success"]
opti_parameter_experiment()

target: 1.5 x: [1.5 1. ] status: 0


### Run your own experiment

The default above is already computed. Change the controls, predict the result, and press **Run experiment**. **Reset controls** restores the initial settings. Solver calls are explicit; dragging controls does not launch optimizations. Python controls require a live JupyterLab kernel.

In [19]:
from casadi_tutorial.interactive import experiment
try:
    import ipywidgets as widgets
except ImportError:
    print("Install the course environment to enable live controls.")
else:
    experiment(opti_parameter_experiment, dict(target_value=widgets.FloatSlider(value=1.5,min=0,max=4,step=.25)), title='Parameterized Opti solve')

### Solve a nonlinear objective with SQP

Sequential quadratic programming repeatedly solves a quadratic approximation to
the nonlinear problem. The Rosenbrock valley has a curved geometry, so its
quadratic model changes after each step. This small unconstrained example uses
CasADi's SQP implementation with QRQP subproblems. Compare the final objective
with zero; do not expect the same iteration trace as IPOPT.

In [14]:
sqp_x=ca.SX.sym("sqp_x",2)
sqp_objective=(1-sqp_x[0])**2+100*(sqp_x[1]-sqp_x[0]**2)**2
sqp_solver=ca.nlpsol("teaching_sqp","sqpmethod",{"x":sqp_x,"f":sqp_objective},
                     {"qpsol":"qrqp","print_header":False,"print_iteration":False,
                      "print_status":False,"print_time":False,"max_iter":100,
                      "qpsol_options":{"print_header":False,"print_iter":False}})
sqp_result=sqp_solver(x0=[1.2,1.2])
assert sqp_solver.stats()["success"] and float(sqp_result["f"])<1e-8
print("SQP optimum:",sqp_result["x"],"objective:",float(sqp_result["f"]))

SQP optimum: [1, 1] objective: 2.780386442419886e-20


## Recap and independent practice

State the mathematical object constructed in this module, the numerical assumptions it requires, and one check that could reveal a wrong result. Change one experimental setting at a time and explain the observed change before continuing.